## Tiktok

#### Useful links:
- [How to set an app for Research API](https://developers.tiktok.com/products/research-api/)
- [Research API Documentation](https://developers.tiktok.com/doc/about-research-api)
- [Access Token Management](https://developers.tiktok.com/doc/client-access-token-management)

In [35]:
# requirements
%pip install pandas requests python-dotenv jmespath

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [139]:
import os
import json
import pandas as pd
import jmespath
import requests
from dotenv import load_dotenv
from datetime import datetime
from time import sleep

In [170]:
# environment variables
load_dotenv()
TIKTOK_CLIENT_KEY = os.getenv("TIKTOK_CLIENT_KEY")
TIKTOK_CLIENT_SECRET = os.getenv("TIKTOK_CLIENT_SECRET")

#### API Access Token & How to connect

In [194]:
# GET Bearer Token
url = "https://open.tiktokapis.com/v2/oauth/token/"

headers = {
    "Content-Type": "application/x-www-form-urlencoded",
    "Cache-Control": "no-cache"
}

data = {
    "client_key": TIKTOK_CLIENT_KEY,
    "client_secret": TIKTOK_CLIENT_SECRET,
    "grant_type": "client_credentials"
}
response = requests.post(url, headers=headers, json=data)
print(response.status_code)

response_json = response.json()

access_token = response_json.get("access_token", None)
token_expiration = response_json.get("expires_in", None) #In seconds. It is valid for 2 hours after the initial issuance.

200


In [ ]:
### TESTING CONNECTION ###

# GET BEARER TOKEN
url = "https://open.tiktokapis.com/v2/oauth/token/"

headers = {
    "Content-Type": "application/x-www-form-urlencoded",
    "Cache-Control": "no-cache",
}

# DATA MUST BE SENT AS FORM-ENCODED, NOT JSON
data = {
    "client_key": TIKTOK_CLIENT_KEY,
    "client_secret": TIKTOK_CLIENT_SECRET,
    "grant_type": "client_credentials",
}


response = requests.post(url, headers=headers, data=data)

print("STATUS CODE:", response.status_code)
print("RAW RESPONSE TEXT:", response.text)

# CHECK FOR SUCCESS BEFORE PARSING JSON FIELDS
try:
    response_json = response.json()
except ValueError:
    # RESPONSE WAS NOT VALID JSON
    response_json = {}
    print("FAILED TO PARSE JSON RESPONSE")

# EXTRACT ACCESS TOKEN AND EXPIRATION IF PRESENT
access_token = response_json.get("access_token")
token_expiration = response_json.get("expires_in")  # IN SECONDS

print("ACCESS TOKEN:", access_token)
print("EXPIRES IN (SECONDS):", token_expiration)

In [148]:
# BASE URL
base_url = "https://open.tiktokapis.com/v2/research/"

In [149]:
# API requests
def request_to_api(access_token, endpoint_url, payload):
    headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}
    response = requests.post(endpoint_url, headers=headers, json=payload)
    return response.json()

# Pagination
def paginate(access_token, endpoint_url, payload, response):
    collected_data = []
    while jmespath.search("data.has_more", response):
        cursor = jmespath.search("data.cursor", response)
        search_id = jmespath.search("data.search_id", response)
        payload.update({"cursor": cursor, "search_id": search_id})                                                                  
        sleep(1) 
        response = request_to_api(access_token, endpoint_url, payload)
        data = response.get("data", [])
        collected_data.extend(data)
    return collected_data


#### Payload

In [145]:
payload_example = {
    "query": {
        "and": [
            {"operation": "IN", "field_name": "region_code", "field_values": ["US", "CA"]},
            {"operation": "EQ", "field_name": "keyword", "field_values": ["hello world"]}
        ]
    },
    "start_date": "20250901", #format: YYYYMMDD
    "end_date": "20250910", #format: YYYYMMDD
    "max_count": 10
}

### SPECIAL CRITERIA

**SC1: Does the platform offer an API for collecting public user-generated content data?**

*This item verifies whether the platform provides an API with at least one endpoint for programmatically extracting public user-generated content to the users’ infrastructure. Public user-generated content is defined here as any publicly visible publication accessible by any platform user. The assessment should verify that the endpoint allows retrieval and storage of this content without requiring privileged or internal access beyond standard developer registration.*

In [54]:
from datetime import datetime, timedelta
if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=7)

payload = {
    "query": {
        "and": [
            {"operation": "IN", "field_name": "region_code", "field_values": ["DE"]},
            {"operation": "EQ", "field_name": "keyword", "field_values": ["taylor swift"]},
        ]
    },
    "start_date": start_date.strftime("%Y%m%d"),
    "end_date": end_date.strftime("%Y%m%d"),
    "max_count": 5,
}

response = request_to_api(access_token, full_url, payload)

videos = response.get("data", {})
if isinstance(videos, dict):
    videos = videos.get("videos") or videos.get("data") or []
elif not isinstance(videos, list):
    videos = []

print(json.dumps(
    {
        "endpoint_details": {"method": "POST", "endpoint": endpoint_suffix},
        "fields": fields.split(","),
        "search_parameters": payload["query"]["and"],
        "sample_video": videos[:3],
        "raw_response_error": response.get("error"),
    },
    indent=2,
))

{
  "endpoint_details": {
    "method": "POST",
    "endpoint": "video/query"
  },
  "fields": [
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username"
  ],
  "search_parameters": [
    {
      "operation": "IN",
      "field_name": "region_code",
      "field_values": [
        "DE"
      ]
    },
    {
      "operation": "EQ",
      "field_name": "keyword",
      "field_values": [
        "taylor swift"
      ]
    }
  ],
  "sample_video": [
    {
      "id": 7579393127262309654,
      "like_count": 33,
      "region_code": "DE",
      "share_count": 0,
      "username": "marinamtr8l2",
      "video_description": "from the Taylor Swift Holiday Collection 2025 #swiftmas #holidaycollection #taylorswift #taylorswiftornaments #taylornation ",
      "comment_count": 0,
      "create_time": 1764714987
    },
    {
      "id": 7579385780481674518,
      "like_count": 182,
      "region_code": "DE",
  

### ACCESSIBILITY


**OC1: Does the platform offer any type of access to non-public data (e.g., private groups, not direct messages) to approved researchers?**

*This item verifies whether, beyond the retrieval and extraction of public user-generated content data, the API permits the extraction of any data from non-public content, such as posts and comments in private groups or discussion forums, subject to the implementation of adequate data hashing measures and specific researcher approval. The assessment should confirm that the API provides such access measures, either through specific endpoints or other controlled access mechanisms.*

In [ ]:
# No, the Tiktok Research API only allows access to public data.

**OC2: Can the requested data be extracted directly from the platform’s API response?**

*This item verifies whether the API returns structured data directly in its response, rather than only providing a redirect link to the data. Audiovisual media (e.g., images, videos, and audio) are excluded from this assessment. The assessment should check sample API responses to confirm that the requested public data is included in the returned payload itself.*


In [ ]:
if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=7)

payload = {
    "query": {
        "and": [
            {"operation": "IN", "field_name": "region_code", "field_values": ["IT"]},
            {"operation": "EQ", "field_name": "keyword", "field_values": ["Harry Potter"]},
        ]
    },
    "start_date": start_date.strftime("%Y%m%d"),
    "end_date": end_date.strftime("%Y%m%d"),
    "max_count": 5,
}

response = request_to_api(access_token, full_url, payload)

videos = response.get("data", {})
if isinstance(videos, dict):
    videos = videos.get("videos") or videos.get("data") or []
elif not isinstance(videos, list):
    videos = []

print(json.dumps(
    {
        "endpoint_details": {"method": "POST", "endpoint": endpoint_suffix},
        "fields": fields.split(","),
        "search_parameters": payload["query"]["and"],
        "sample_video": videos[:1],
        "raw_response_error": response.get("error"),
    },
    indent=2,
))

{
  "endpoint_details": {
    "method": "POST",
    "endpoint": "video/query"
  },
  "fields": [
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username"
  ],
  "search_parameters": [
    {
      "operation": "IN",
      "field_name": "region_code",
      "field_values": [
        "IT"
      ]
    },
    {
      "operation": "EQ",
      "field_name": "keyword",
      "field_values": [
        "Harry Potter"
      ]
    }
  ],
  "sample_video": [
    {
      "comment_count": 1,
      "create_time": 1764713679,
      "id": 7579387515258342658,
      "like_count": 1,
      "region_code": "IT",
      "share_count": 0,
      "username": "jessica7c",
      "video_description": "Per noi amanti di Harry Potter...\u2728\ufe0f"
    }
  ],
  "raw_response_error": {
    "code": "ok",
    "message": "",
    "log_id": "20251205161454B605C14FEE09FB0EF907"
  }
}


**OC4: Does the platform’s API offer an endpoint for extracting data from an individual publication?**

*This item verifies whether it is possible to collect data from a specific public post on the platform using a unique identifier, rather than relying on search terms or other filters. The assessment should review the API documentation and test available endpoints to confirm that an individual publication can be retrieved directly by its unique identifier.*


In [ ]:
# Use this cell to develop an example where a request for a specific post is made (by its id).
# Please leave the result as the output of this cell. 
post_id = 7578660067394735363 # SET THE VALUE HERE
endpoint_suffix = "video/query" # SET correct value (e.g "video/query")
fields = "region_code,music_id,share_count" # SET correct value (e.g "id,like_count")
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"
payload = {
    "query": {
        "and": [
            {"operation": "EQ", "field_name": "video_id", "field_values": [str(post_id)]}
        ]
    },
    "start_date": "20251115",  # widen as needed; YYYYMMDD
    "end_date":   "20251215",
    "max_count": 10
}

response = request_to_api(access_token, full_url, payload)
print(json.dumps(response, indent=2))


{
  "data": {
    "cursor": 1,
    "has_more": false,
    "search_id": "7579990792282903607",
    "videos": [
      {
        "music_id": 7578660077750569750,
        "region_code": "IT",
        "share_count": 1
      }
    ]
  },
  "error": {
    "code": "ok",
    "message": "",
    "log_id": "20251204131701A86950E373F0BA05B241"
  }
}


**OC5: Does the platform’s API offer an endpoint for extracting data from an individual author?**

*This item verifies whether it is possible to collect data from public posts made by a specific author, using their username or unique identifier. The assessment should review the API documentation and test relevant endpoints to confirm that data can be retrieved directly for an individual author.*


In [57]:
# Use this cell to develop an example where a request for a specific author is made.
# Please leave the result as the output of this cell.
author = "taylorswift" # SET THE VALUE HERE
endpoint_suffix = "video/query" # SET correct value (e.g "video/query")
fields = "id,video_description,view_count" # SET correct value (e.g "id,like_count")
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"


payload = {
    "query": {
        "and": [
             {"operation": "EQ", "field_name": "username", "field_values": [author]},
        ]
    },
    "start_date": "20251125",  # widen as needed; YYYYMMDD
    "end_date":   "20251205",
    "max_count": 10
}

response = request_to_api(access_token, full_url, payload)
print(json.dumps(response, indent=2))

{
  "data": {
    "has_more": false,
    "videos": [],
    "cursor": 0
  },
  "error": {
    "code": "ok",
    "message": "",
    "log_id": "20251205173038A31C1927F254C7124AD9"
  }
}


This does not appear to work, therefore I believe that there are no workable workarounds to this issue.

**OC6: Does the platform’s API provide an endpoint for extracting data based on search terms?**

*This item verifies whether public user-generated content can be collected via individual or combined search terms, enabling the creation of datasets of posts mentioning those terms. The assessment should test search-related endpoints to confirm that queries using keywords return relevant public posts.*


In [66]:
endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=7)  # adjust if you want a wider window

payload = {
    "query": {
        "and": [
            {"operation": "IN", "field_name": "region_code", "field_values": ["DE"]},  # adjust regions as needed
            {"operation": "EQ", "field_name": "keyword", "field_values": ["star wars"]},
        ]
    },
    "start_date": start_date.strftime("%Y%m%d"),
    "end_date": end_date.strftime("%Y%m%d"),
    "max_count": 5,
}

response = request_to_api(access_token, full_url, payload)

videos = response.get("data", {})
if isinstance(videos, dict):
    videos = videos.get("videos") or videos.get("data") or []
elif not isinstance(videos, list):
    videos = []

print(json.dumps(
    {
        "endpoint_details": {"method": "POST", "endpoint": endpoint_suffix},
        "fields": fields.split(","),
        "search_parameters": payload["query"]["and"],
        "videos_found": len(videos),
        "sample_videos": videos[:3],  # first 3; change slice or drop it to print all
        "raw_response_error": response.get("error"),
    },
    indent=2,
))

{
  "endpoint_details": {
    "method": "POST",
    "endpoint": "video/query"
  },
  "fields": [
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username"
  ],
  "search_parameters": [
    {
      "operation": "IN",
      "field_name": "region_code",
      "field_values": [
        "DE"
      ]
    },
    {
      "operation": "EQ",
      "field_name": "keyword",
      "field_values": [
        "star wars"
      ]
    }
  ],
  "videos_found": 5,
  "sample_videos": [
    {
      "create_time": 1764887096,
      "id": 7580132340790250774,
      "like_count": 151,
      "region_code": "DE",
      "share_count": 2,
      "username": "itsstewie8",
      "video_description": "Star Wars The Clone Wars - Obi-Wan Kenobi & Anakin Skywalker Vs. Asajj Ventress.                                                #starwars #starwarstheclonewars #clonewars #anakinskywalker #obiwan ",
      "comment_count": 0
    },
   

Yes, this API provides an endpoint for extracting data based on search terms

**OC7: Does the API use locale-neutral data representations?**

*This item verifies whether locale-sensitive data (e.g., timestamps, currency, numbers) are returned in a locale-neutral format, or whether relevant locale metadata is included when neutrality is not possible. The assessment should review the API documentation and inspect sample responses to confirm the presence of standardized formats or accompanying metadata.*


In [72]:
endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "create_time",
    "region_code",
    "share_count",
    "view_count",
    "like_count",
    "comment_count",
    "favorites_count",
    "video_duration",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=7)  # widen/narrow as needed

payload = {
    "query": {
        "and": [
            {"operation": "IN", "field_name": "region_code", "field_values": ["FR"]},
            {"operation": "EQ", "field_name": "keyword", "field_values": ["Disney"]},
        ]
    },
    "start_date": start_date.strftime("%Y%m%d"),
    "end_date": end_date.strftime("%Y%m%d"),
    "max_count": 5,
}

resp = request_to_api(access_token, full_url, payload)
videos = resp.get("data", {}).get("videos", []) or []

print(json.dumps({
    "error": resp.get("error"),
    "videos_found": len(videos),
    "sample_videos": videos[:5],  # first 5 (or fewer if not enough matches)
}, indent=2))

{
  "error": {
    "code": "ok",
    "message": "",
    "log_id": "2025120722382579140EECF4808F820B0B"
  },
  "videos_found": 4,
  "sample_videos": [
    {
      "create_time": 1764892346,
      "id": 7580154847123492118,
      "region_code": "FR",
      "comment_count": 732,
      "favorites_count": 36613,
      "like_count": 301591,
      "share_count": 26301,
      "video_duration": 43,
      "view_count": 1661339
    },
    {
      "region_code": "FR",
      "video_duration": 6,
      "view_count": 2700,
      "create_time": 1764892062,
      "favorites_count": 19,
      "id": 7580153633350634774,
      "like_count": 357,
      "share_count": 57,
      "comment_count": 4
    },
    {
      "favorites_count": 3,
      "id": 7580152690290838806,
      "like_count": 5,
      "view_count": 323,
      "comment_count": 0,
      "create_time": 1764891841,
      "region_code": "FR",
      "share_count": 1,
      "video_duration": 226
    },
    {
      "share_count": 0,
      "video_durati

In [ ]:
import re
from datetime import datetime, timezone

videos = resp.get("data", {}).get("videos", []) or []

iso8601 = re.compile(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(\.\d+)?(Z|[+-]\d{2}:\d{2}|[+-]\d{4})$")
numeric_fields = ["like_count", "comment_count", "share_count", "view_count", "favorites_count"]

def to_iso(ts):
    if isinstance(ts, (int, float)):
        return datetime.fromtimestamp(ts, tz=timezone.utc).isoformat().replace("+00:00", "Z")
    return ts

def analyze_video(v):
    ts = v.get("create_time")
    ts_iso = to_iso(ts)
    ts_is_iso = isinstance(ts_iso, str) and bool(iso8601.match(ts_iso))
    ts_is_epoch = isinstance(ts, (int, float))
    numeric_ok = all(isinstance(v.get(f), (int, float, type(None))) for f in numeric_fields)
    return {
        "id": v.get("id"),
        "create_time_raw": ts,
        "create_time_iso": ts_iso,
        "create_time_is_iso8601": ts_is_iso,
        "create_time_is_epoch": ts_is_epoch,
        "numeric_fields_are_numbers": numeric_ok,
    }

analysis = [analyze_video(v) for v in videos[:5]]

print(json.dumps({
    "error": resp.get("error"),
    "videos_found": len(videos),
    "sample_analysis": analysis,
}, indent=2))

{
  "error": {
    "code": "ok",
    "message": "",
    "log_id": "2025120722382579140EECF4808F820B0B"
  },
  "videos_found": 4,
  "sample_analysis": [
    {
      "id": 7580154847123492118,
      "create_time_raw": 1764892346,
      "create_time_iso": "2025-12-04T23:52:26Z",
      "create_time_is_iso8601": true,
      "create_time_is_epoch": true,
      "numeric_fields_are_numbers": true
    },
    {
      "id": 7580153633350634774,
      "create_time_raw": 1764892062,
      "create_time_iso": "2025-12-04T23:47:42Z",
      "create_time_is_iso8601": true,
      "create_time_is_epoch": true,
      "numeric_fields_are_numbers": true
    },
    {
      "id": 7580152690290838806,
      "create_time_raw": 1764891841,
      "create_time_iso": "2025-12-04T23:44:01Z",
      "create_time_is_iso8601": true,
      "create_time_is_epoch": true,
      "numeric_fields_are_numbers": true
    },
    {
      "id": 7580152561265544470,
      "create_time_raw": 1764891812,
      "create_time_iso": "2025-

### COMPLIANCE

**OC15: Does the platform provide a way to label content that has been generated with artificial intelligence?**

*This item verifies whether the platform automatically flags, or allows users to flag, AI-generated content, and whether this information is given in the API response. The assessment should review the documentation and test API outputs to confirm that these flags are included in the extracted data.* 


In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
endpoint_suffix = None # SET correct value (e.g "video/query")
fields = None # SET correct value (e.g "id,like_count")
full_url = f"{base_url}/{endpoint_suffix}/?fields={fields}"
payload = {} # check paylod example
response = request_to_api(access_token, full_url, payload)

In [ ]:
from datetime import datetime, timedelta
import json

if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "create_time",
    "region_code",
    "share_count",
    "view_count",
    "like_count",
    "comment_count",
    "favorites_count",
    "video_duration",
    "video_tag",  # struct array with number/type
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=7)

payload = {
    "query": {
        "and": [
            {"operation": "EQ", "field_name": "keyword", "field_values": ["Trump"]},
        ]
    },
    "start_date": start_date.strftime("%Y%m%d"),
    "end_date": end_date.strftime("%Y%m%d"),
    "max_count": 10,
}

resp = request_to_api(access_token, full_url, payload)
videos = resp.get("data", {}).get("videos", []) or []

# See the video_tag documentation here:
# https://developers.tiktok.com/doc/research-api-specs-query-videos

TAG_MAP = {
    ("AIGC Type", 1): "Creator labelled as AI-generated",
    ("AIGC Type", 2): "AI-generated (system)",
    ("Branded Type", 1): "Paid partnership",
    ("Branded Type", 7): "Creator earns commission",
}

def summarize_tags(tag_list):
    if not isinstance(tag_list, list):
        return []
    out = []
    for t in tag_list:
        ttype = (t or {}).get("type")
        num = (t or {}).get("number")
        label = TAG_MAP.get((ttype, num))
        out.append({"type": ttype, "number": num, "label": label})
    return out

analysis = [
    {
        "id": v.get("id"),
        "video_tag_raw": v.get("video_tag"),
        "video_tag_summary": summarize_tags(v.get("video_tag")),
    }
    for v in videos
]

print(json.dumps({
    "error": resp.get("error"),
    "videos_found": len(videos),
    "sample_videos": videos[:3],       # first 3 full records
    "tag_analysis": analysis[:10],     # tag summaries for up to 10
}, indent=2))


{
  "error": {
    "code": "ok",
    "message": "",
    "log_id": "20251208104039B6B4AD7D1ACFF700F5DE"
  },
  "videos_found": 9,
  "sample_videos": [
    {
      "share_count": 0,
      "create_time": 1764979195,
      "favorites_count": 0,
      "like_count": 3,
      "region_code": "BR",
      "view_count": 139,
      "comment_count": 0,
      "id": 7580527881705606418,
      "video_duration": 131,
      "video_tag": [
        {
          "number": 1,
          "type": "AIGC Type"
        }
      ]
    },
    {
      "comment_count": 5,
      "create_time": 1764979198,
      "favorites_count": 0,
      "like_count": 20,
      "share_count": 1,
      "id": 7580527873010699551,
      "region_code": "US",
      "video_duration": 21,
      "view_count": 354
    },
    {
      "create_time": 1764979198,
      "favorites_count": 0,
      "share_count": 0,
      "video_duration": 1246,
      "comment_count": 1,
      "id": 7580527846091738398,
      "like_count": 0,
      "region_code": "US

One video found to be labelled as AI-generated.

### COMPLETENESS

**OC16: Can data from a publication’s comments be extracted using the platform’s API?**

*This item verifies whether comment data, including their content, can be extracted when available on the platform, either together with publication data or with a dedicated endpoint. The assessment should test relevant endpoints to confirm that comments are retrievable as structured data. This item does not apply to platforms that do not have commenting features.*


In [102]:


video_id = 7580527873010699551  # use an actual video id that has comments (int)

endpoint_suffix = "video/comment/list/"
fields = "id,text,like_count,reply_count,parent_comment_id,create_time,video_id"
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

payload = {"video_id": video_id, "max_count": 20, "cursor": 0}
headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}

r = requests.post(full_url, headers=headers, json=payload)
print("status:", r.status_code, "ct:", r.headers.get("content-type"))
print("preview:", r.text[:400])  # see if it’s HTML/empty/error
r.raise_for_status()
data = r.json()

comments = data.get("data", {}).get("comments", []) or []
print(json.dumps({
    "error": data.get("error"),
    "comments_found": len(comments),
    "comment_texts": [c.get("text") for c in comments[:20]],
}, indent=2))


status: 200 ct: application/json
preview: {"data":{"comments":[{"video_id":7580527873010699551,"create_time":1764988131,"id":7580566214772327223,"like_count":1,"parent_comment_id":7580527873010699551,"reply_count":1,"text":"2nd place"},{"parent_comment_id":7580527873010699551,"reply_count":0,"text":"yup.","video_id":7580527873010699551,"create_time":1764985702,"id":7580555774575149837,"like_count":1},{"id":7580554078309090078,"like_count"
{
  "error": {
    "code": "ok",
    "message": "ok",
    "log_id": "2025120811110782C4BF3D7AB01A01971A"
  },
  "comments_found": 3,
  "comment_texts": [
    "2nd place",
    "yup.",
    "what about the ending of 8 conflicts"
  ]
}


**OC17: Can data from temporary content be extracted through the platform’s API?**

*This item verifies whether the platform’s API provides at least one endpoint for collecting data from temporary publications (e.g., stories, ephemeral messages). The assessment should test endpoints to confirm whether this type of short-lived content can be retrieved as structured data before it expires. This item does not apply to platforms that do not have temporary content features.*


In [ ]:
#No endpoints or fields are provided to access temporary content such as stories. 

**OC18: Can historical data be extracted through the platform’s API?**

*This item verifies whether the API provides endpoints that allow for a specified time range, going back more than one year from the time the request is made, to collect public user-generated content data. The assessment should review test endpoints to confirm that historical data more than 12 months prior to the analysis can be retrieved.*

In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
endpoint_suffix = None # SET correct value (e.g "video/query")
fields = None # SET correct value (e.g "id,like_count")
full_url = f"{base_url}/{endpoint_suffix}/?fields={fields}"
payload = {} # check paylod example
response = request_to_api(access_token, full_url, payload)

In [114]:
from datetime import datetime

if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "create_time",
    "region_code",
    "share_count",
    "view_count",
    "like_count",
    "comment_count",
    "favorites_count",
    "video_duration",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

# Full year 2023
payload = {
    "query": {
        "and": [
            {"operation": "EQ", "field_name": "keyword", "field_values": ["Incredibles"]},
        ]
    },
    "start_date": "20190101",
    "end_date":   "20190131",
    "max_count": 20,  # adjust as needed
}

resp = request_to_api(access_token, full_url, payload)
videos = resp.get("data", {}).get("videos", []) or []

print(json.dumps({
    "error": resp.get("error"),
    "videos_found": len(videos),
    "sample_videos": videos[:5],  # first 5
}, indent=2))


{
  "error": {
    "code": "ok",
    "message": "",
    "log_id": "202512081419464EA4C1180DD84F083907"
  },
  "videos_found": 9,
  "sample_videos": [
    {
      "create_time": 1548973373,
      "favorites_count": 0,
      "share_count": 0,
      "comment_count": 7,
      "id": 6652789980189953285,
      "like_count": 32,
      "region_code": "US",
      "video_description": "My movie trailer for Incredibles 2",
      "video_duration": 0,
      "view_count": 137
    },
    {
      "create_time": 1548972323,
      "video_duration": 0,
      "comment_count": 0,
      "favorites_count": 2,
      "id": 6652785470830808325,
      "like_count": 80,
      "region_code": "GB",
      "share_count": 1,
      "video_description": "#duet with @iambannd #wevebeenplanning #dinner #foryoupage #frozone #frozoneswife #incredibles",
      "view_count": 490
    },
    {
      "like_count": 2,
      "share_count": 0,
      "video_duration": 0,
      "view_count": 22,
      "comment_count": 0,
      "creat

**OC19: Is the number of requests allowed by the API sufficient for monitoring more than 10,000 publications in 24 hours?**

*This item verifies whether data can be extracted without interruption and losses through the platform’s API for requests that accumulate more than 10,000 publications in 24 hours. The assessment should test the API to confirm that this volume of data can be collected continuously.*


In [206]:
import json, time
from datetime import datetime, timedelta
from pathlib import Path

if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
stats_file = data_dir / f"eu-tiktok-UGC-question-19-stats-{timestamp}.json"

endpoint_suffix = "video/query"
fields = "id,create_time,video_description"
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

# Fixed query (adjust keyword/region if needed, keep it consistent)
payload = {
    "query": {"and": [
        {"operation": "EQ", "field_name": "keyword", "field_values": ["music"]},
    ]},
    "start_date": "20251001",
    "end_date":   "20251021",
    "max_count": 50,
}

runs = 5  # short burst for measurement; adjust if you want a longer sample
collected = 0
start = time.time()

for i in range(runs):
    resp = request_to_api(access_token, full_url, payload)
    vids = resp.get("data", {}).get("videos", []) or []
    collected += len(vids)
    print(f"run {i+1}: error={resp.get('error')} videos={len(vids)}")

elapsed = time.time() - start
reqs_per_sec = runs / elapsed if elapsed else 0
items_per_sec = collected / elapsed if elapsed else 0
items_per_24h = int(items_per_sec * 86400)

stats = {
    "runs": runs,
    "total_requests": runs,
    "total_videos_collected": collected,
    "elapsed_seconds": elapsed,
    "requests_per_second": reqs_per_sec,
    "videos_per_second": items_per_sec,
    "videos_per_24h_extrapolated": items_per_24h,
    "query": payload,
    "fields": fields.split(","),
    "endpoint": endpoint_suffix,
    "timestamp_utc": timestamp,
}

with stats_file.open("w") as fout:
    json.dump(stats, fout, indent=2)

print("\nSaved stats to", stats_file)
print(json.dumps({k: stats[k] for k in ["total_requests","total_videos_collected","requests_per_second","videos_per_24h_extrapolated"]}, indent=2))


run 1: error={'code': 'ok', 'message': '', 'log_id': '202512101816431DDB353C133F37110C09'} videos=40
run 2: error={'code': 'ok', 'message': '', 'log_id': '20251210181653C4BDC268F9BA14137A1E'} videos=40
run 3: error={'code': 'ok', 'message': '', 'log_id': '20251210181657C4BDC268F9BA14137AF6'} videos=40
run 4: error={'code': 'ok', 'message': '', 'log_id': '20251210181701093AEE04C06F75127F9B'} videos=40
run 5: error={'code': 'ok', 'message': '', 'log_id': '202512101817021DDB353C133F37110FA0'} videos=40

Saved stats to data/eu-tiktok-UGC-question-19-stats-20251210T181643Z.json
{
  "total_requests": 5,
  "total_videos_collected": 200,
  "requests_per_second": 0.2578091930061792,
  "videos_per_24h_extrapolated": 890988
}


In [210]:
import json, time
from datetime import datetime
from pathlib import Path

if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = "id,video_description,username,create_time,region_code,like_count,comment_count,share_count,view_count"
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

payload = {
    "query": {"and": [
        {"operation": "EQ", "field_name": "keyword", "field_values": ["climate"]},  # adjust if desired
    ]},
    "start_date": "20251001",
    "end_date":   "20251021",
    "max_count": 50,
}

#Videos per second ≈ 10.3 (890,988 per 24h ÷ 86,400).
#Time to hit 10,000 videos ≈ 10,000 ÷ 10.3 ≈ 971 seconds ≈ 16.2 minutes.

time_limit_seconds = 30  # 17 minutes based on calucations above 
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
ts = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
data_file = data_dir / f"eu-tiktok-UGC-question-19-data-{ts}.jsonl"
stats_file = data_dir / f"eu-tiktok-UGC-question-19-stats-{ts}.json"

total_videos = 0
total_requests = 0
start = time.time()
cursor = None

with data_file.open("w") as fout:
    while time.time() - start < time_limit_seconds:
        if cursor:
            payload["cursor"] = cursor
        resp = request_to_api(access_token, full_url, payload)
        total_requests += 1
        vids = resp.get("data", {}).get("videos", []) or []
        for v in vids:
            json.dump(v, fout)
            fout.write("\n")
        total_videos += len(vids)
        cursor = resp.get("data", {}).get("cursor")
        has_more = resp.get("data", {}).get("has_more")
        # Stop paging if no more; optionally restart a new query to keep collecting
        if not has_more:
            cursor = None  # reset to start a fresh query
        # Minimal pacing to be gentle; adjust or remove if allowed
        time.sleep(0.5)

elapsed = time.time() - start
stats = {
    "runs": total_requests,
    "total_requests": total_requests,
    "total_videos_collected": total_videos,
    "elapsed_seconds": elapsed,
    "requests_per_second": total_requests / elapsed if elapsed else 0,
    "videos_per_second": total_videos / elapsed if elapsed else 0,
    "videos_per_24h_extrapolated": int((total_videos / elapsed) * 86400) if elapsed else 0,
    "endpoint": endpoint_suffix,
    "fields": fields.split(","),
    "query": payload.get("query"),
    "timestamp_utc": ts,
    "data_file": str(data_file),
}
with stats_file.open("w") as f:
    json.dump(stats, f, indent=2)

print("Done. Stats:", json.dumps({k: stats[k] for k in [
    "total_requests", "total_videos_collected", "requests_per_second", "videos_per_24h_extrapolated"
]}, indent=2))
print("Saved data to", data_file)
print("Saved stats to", stats_file)


Done. Stats: {
  "total_requests": 40,
  "total_videos_collected": 0,
  "requests_per_second": 1.3329792115761077,
  "videos_per_24h_extrapolated": 0
}
Saved data to data/eu-tiktok-UGC-question-19-data-20251210T192358Z.jsonl
Saved stats to data/eu-tiktok-UGC-question-19-stats-20251210T192358Z.json


In [211]:
# Inspect one response to see the error block and has_more
resp = request_to_api(access_token, full_url, payload)
print("error:", resp.get("error"))
print("has_more:", resp.get("data", {}).get("has_more"))
print("videos:", len(resp.get("data", {}).get("videos", []) or []))


error: {'code': 'daily_quota_limit_exceeded', 'message': 'The API daily quota limit exceeded. Please try again later', 'log_id': '20251210192541B3724DC9FA7D75164980'}
has_more: None
videos: 0


### CONSISTENCY

**OC20: Are the results returned by the API consistently reproducible?**

*This item verifies whether data extracted via the platform’s API at any given time is consistent with other collections performed similarly, including content that has been deleted between collections. The assessment should conduct repeated test queries to confirm the reproducibility of results or ground the response based on recent (less than 2 years) experiments published in peer-reviewed journals.*


Test instructions: Please, develop a test that collects data for 5 times using the same request parameters and the same endpoint. Save each response in separate files.

In [134]:
import json
from pathlib import Path
from datetime import datetime

if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "username",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

# Fixed parameters for all runs
payload = {
    "query": {
        "and": [
            {"operation": "EQ", "field_name": "keyword", "field_values": ["Incredibles"]},
        ]
    },
    "start_date": "20230101",
    "end_date": "20230131",
    "max_count": 10,
}

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

for idx in range(5):
    resp = request_to_api(access_token, full_url, payload)
    out_path = data_dir / f"tiktok-consistency-run-{idx+1}-{timestamp}.json"
    with out_path.open("w") as fout:
        json.dump(resp, fout, indent=2)
    print(f"Saved run {idx+1} to {out_path}")


Saved run 1 to data/tiktok-consistency-run-1-20251208T155210Z.json
Saved run 2 to data/tiktok-consistency-run-2-20251208T155210Z.json
Saved run 3 to data/tiktok-consistency-run-3-20251208T155210Z.json
Saved run 4 to data/tiktok-consistency-run-4-20251208T155210Z.json
Saved run 5 to data/tiktok-consistency-run-5-20251208T155210Z.json


In [136]:
import json, glob
for path in sorted(glob.glob("data/tiktok-consistency-run-*.json")):
    with open(path) as f:
        data = json.load(f)
    vids = data.get("data", {}).get("videos", []) or []
    print(path, "videos:", len(vids), "sample_id:", vids[0].get("id") if vids else None)


data/tiktok-consistency-run-1-20251208T155210Z.json videos: 7 sample_id: 7194964582463540482
data/tiktok-consistency-run-2-20251208T155210Z.json videos: 7 sample_id: 7194964582463540482
data/tiktok-consistency-run-3-20251208T155210Z.json videos: 7 sample_id: 7194964582463540482
data/tiktok-consistency-run-4-20251208T155210Z.json videos: 7 sample_id: 7194964582463540482
data/tiktok-consistency-run-5-20251208T155210Z.json videos: 7 sample_id: 7194964582463540482


**OC21: Is the data returned by the platform’s API consistent with the parameters and filters used in the request?**

*This item verifies whether the data extracted through the API accurately reflects the parameters and filters specified in the request. The assessment should conduct repeated test queries to confirm the consistency of results or ground the response based on recent (less than 2 years) experiments published in peer-reviewed journals.*


Test instructions: Please, develop a process that request data using different parameter to the same endpoint. If possible, test at least 5 different parameters/filters. If the platform provides less than 5, use all available parameters/filters.

Save each response in separate files. 

In [ ]:
#3rd retry

from pathlib import Path
from datetime import datetime, timedelta

if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "view_count",
    "username",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

# Fixed date window
end_date = datetime.utcnow()
start_date = end_date - timedelta(days=30)

# Five different filters/parameters against the same endpoint
parameter_sets = [
    {"label": "kw_incredibles", "query": {"and": [
        {"operation": "EQ", "field_name": "keyword", "field_values": ["Incredibles"]},
    ]}},
    {"label": "kw_wicked", "query": {"and": [
        {"operation": "EQ", "field_name": "keyword", "field_values": ["Wicked"]},
    ]}},
    {"label": "kw_pixar", "query": {"and": [
        {"operation": "EQ", "field_name": "keyword", "field_values": ["Pixar"]},
    ]}},
    {"label": "region_fr_kw_film", "query": {"and": [
        {"operation": "IN", "field_name": "region_code", "field_values": ["FR"]},
        {"operation": "EQ", "field_name": "keyword", "field_values": ["film"]},
    ]}},
    {"label": "region_de_kw_music", "query": {"and": [
        {"operation": "IN", "field_name": "region_code", "field_values": ["DE"]},
        {"operation": "EQ", "field_name": "keyword", "field_values": ["music"]},
    ]}},
]

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

def build_payload(query_block):
    return {
        "query": query_block,
        "start_date": start_date.strftime("%Y%m%d"),
        "end_date": end_date.strftime("%Y%m%d"),
        "max_count": 20,
    }

for idx, param in enumerate(parameter_sets, start=1):
    payload = build_payload(param["query"])
    resp = request_to_api(access_token, full_url, payload)
    filename = f"tiktok-UGC-question-21-run-{idx}-{param['label']}-{timestamp}.json"
    out_path = data_dir / filename
    with out_path.open("w") as fout:
        json.dump(resp, fout, indent=2)
    vids = resp.get("data", {}).get("videos", []) or []
    print(f"Saved {out_path} | videos: {len(vids)} | sample_id: {vids[0].get('id') if vids else None}")





Saved data/tiktok-UGC-question-21-run-1-kw_incredibles-20251210T140103Z.json | videos: 19 | sample_id: 7581185391546387719
Saved data/tiktok-UGC-question-21-run-2-kw_wicked-20251210T140103Z.json | videos: 15 | sample_id: 7581269314376961298
Saved data/tiktok-UGC-question-21-run-3-kw_pixar-20251210T140103Z.json | videos: 18 | sample_id: 7581248417398099218
Saved data/tiktok-UGC-question-21-run-4-region_fr_kw_film-20251210T140103Z.json | videos: 19 | sample_id: 7581259735626698006
Saved data/tiktok-UGC-question-21-run-5-region_de_kw_music-20251210T140103Z.json | videos: 19 | sample_id: 7581258083226144022


In [162]:
import json, glob
for path in sorted(glob.glob("data/tiktok-UGC-question-21-run-*.json")):
    with open(path) as f:
        resp = json.load(f)
    vids = resp.get("data", {}).get("videos", []) or []
    err = resp.get("error")
    has_more = resp.get("data", {}).get("has_more")
    print(path, "error:", err, "videos:", len(vids), "has_more:", has_more, "sample_id:", vids[0].get("id") if vids else None)


data/tiktok-UGC-question-21-run-1-kw_incredibles-20251210T140103Z.json error: {'code': 'ok', 'message': '', 'log_id': '202512101401039B05D4BA1117A20804C5'} videos: 19 has_more: True sample_id: 7581185391546387719
data/tiktok-UGC-question-21-run-2-kw_wicked-20251210T140103Z.json error: {'code': 'ok', 'message': '', 'log_id': '20251210140105E297EEF84CC09308BB38'} videos: 15 has_more: True sample_id: 7581269314376961298
data/tiktok-UGC-question-21-run-3-kw_pixar-20251210T140103Z.json error: {'code': 'ok', 'message': '', 'log_id': '20251210140110DB8EF6D005DF300849C7'} videos: 18 has_more: True sample_id: 7581248417398099218
data/tiktok-UGC-question-21-run-4-region_fr_kw_film-20251210T140103Z.json error: {'code': 'ok', 'message': '', 'log_id': '20251210140112E297EEF84CC09308BCA8'} videos: 19 has_more: True sample_id: 7581259735626698006
data/tiktok-UGC-question-21-run-5-region_de_kw_music-20251210T140103Z.json error: {'code': 'ok', 'message': '', 'log_id': '202512101401141952F5CB6399FE07A51

In [168]:
import json, glob

for path in sorted(glob.glob("/Users/anitac/projects/transparency-index/2025/user-generated-content-framework/EU/data/tiktok-UGC-question-21-run-*.json")):
    with open(path) as f:
        data = json.load(f)
    videos = data.get("data", {}).get("videos", []) or []
    print(f"\n{filename} | videos: {len(videos)}")
    for v in videos[:3]:
        print("  - id:", v.get("id"), "| desc:", (v.get("video_description") or "").strip()[:160])



tiktok-UGC-question-21-run-5-region_de_kw_music-20251210T140103Z.json | videos: 19
  - id: 7581185391546387719 | desc: Bettama Edit #gammajack #Edit #increible #incredibles #Viral
  - id: 7581134089592294670 | desc: PLEASE DON'T HATE I WANNA MAKE THIS AS INCREDIBLES x LOUD HOUSE #LOUDHOUSE
  - id: 7581108973227658503 | desc: #dueto #fypviralシ #incredibles

tiktok-UGC-question-21-run-5-region_de_kw_music-20251210T140103Z.json | videos: 15
  - id: 7581269314376961298 | desc: David Corenswet faz comentário sem noção sobre a masculinidade do Jonathan Bailey em uma cena com Cynthia Erivo e causa revolta na internet! #davidcorenswet #jo
  - id: 7581268205608471838 | desc: Day 6 advent calendars. Honestly didn’t post yesterday because I looked like crap and felt like crap lol. Your girls still on her monthly gift and the cramps ar
  - id: 7581259757852413205 | desc: Esto es lo peor que le puede pasar a alguien que colecciona fragancias. Me volvió a pasar con mi body spray  Que puedo hacer ? 

### RELEVANCE

**OC22: Does the data extracted by the platform’s API reflect what is displayed on its user interface?**

*This item verifies whether the data returned by the API corresponds to the information displayed on the platform’s user interface at all levels of detail. The assessment should compare API responses with the user interface to confirm that key elements—such as authorship, complete content, interaction counts (e.g., comments, shares, replies), and referenced content (e.g., shares, mentions)—are fully represented.*


In [ ]:
#2nd try

# Step 1: finding a public video id (keyword + region BE)
search_payload = {
    "query": {"and": [
        {"operation": "EQ", "field_name": "keyword", "field_values": ["Noel"]},
        {"operation": "IN", "field_name": "region_code", "field_values": ["FR"]},
    ]},
    "start_date": "20251120",
    "end_date":   "20251205",
    "max_count": 5,
}
search_resp = request_to_api(access_token, f"{base_url}video/query?fields=id", search_payload)
vids = search_resp.get("data", {}).get("videos", []) or []
print("candidate ids:", [v.get("id") for v in vids])

# Step 2: testing the id to be compared with the UI at tiktok.com/[username]/video/[video_id]
if vids:
    video_id = str(vids[0]["id"])
    by_id_payload = {
        "query": {"and": [
            {"operation": "EQ", "field_name": "video_id", "field_values": [video_id]},
        ]},
        "start_date": "20251120",
        "end_date":   "20251205",
        "max_count": 1,
    }
    resp = request_to_api(
        access_token,
        f"{base_url}video/query?fields=id,video_description,username,create_time,like_count,comment_count,share_count,view_count",
        by_id_payload,
    )
    print("error:", resp.get("error"))
    print("data:", resp.get("data"))
else:
    print("No candidates found; try widening dates/keyword or removing region filter.")


candidate ids: [7580527875959377174, 7580527870007676182, 7580527693871975702, 7580527617774783766, 7580527594622471427]
error: {'code': 'ok', 'message': '', 'log_id': '20251210161324384C60B77FCA850F0ACD'}
data: {'cursor': 1, 'has_more': False, 'search_id': '7582258536626000910', 'videos': [{'id': 7580527875959377174, 'like_count': 3, 'share_count': 0, 'username': 'sunny.5672', 'video_description': '#paris #france #noel #lumiere #bynight ', 'view_count': 467, 'comment_count': 0, 'create_time': 1764979194}]}


In [188]:
    videos = resp.get("data", {}).get("videos", []) or []
    if not videos:
        print("No video returned for", video_id)
    else:
        v = videos[0]
        print("\n--- VIDEO DETAILS ---")
        print("id:", v.get("id"))
        print("author:", v.get("username"))
        print("caption:", v.get("video_description"))
        print("create_time:", v.get("create_time"))
        print("counts -> likes:", v.get("like_count"),
              "comments:", v.get("comment_count"),
              "shares:", v.get("share_count"),
              "views:", v.get("view_count"))



--- VIDEO DETAILS ---
id: 7580527875959377174
author: sunny.5672
caption: #paris #france #noel #lumiere #bynight 
create_time: 1764979194
counts -> likes: 3 comments: 0 shares: 0 views: 467


This matches the details of this public video visible on the UI at https://www.tiktok.com/@sunny.5672/video/7580527875959377174, verified on 10/12/25.

**OC23: Does the platform’s API allow for filtering data based on publisher location?**

*This item verifies whether the API supports applying location-based filters to data extraction. The assessment should test the endpoint for the main content type to confirm that data on public posts can be filtered by publisher location.*


In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
location = None # SET location
endpoint_suffix = None # SET correct value (e.g "video/query")
fields = None # SET correct value (e.g "id,like_count")
full_url = f"{base_url}/{endpoint_suffix}/?fields={fields}"
payload = {} # check paylod example
response = request_to_api(access_token, full_url, payload)

In [192]:
endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "username",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "view_count",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=30)
keyword = "Navidad"  # set to None to drop keyword filter

# Build query
and_filters = [
    {"operation": "IN", "field_name": "region_code", "field_values": ["ES"]},
]
if keyword:
    and_filters.append({"operation": "EQ", "field_name": "keyword", "field_values": [keyword]})

payload = {
    "query": {"and": and_filters},
    "start_date": start_date.strftime("%Y%m%d"),
    "end_date": end_date.strftime("%Y%m%d"),
    "max_count": 5,
}

resp = request_to_api(access_token, full_url, payload)
videos = resp.get("data", {}).get("videos", []) or []

print(json.dumps({
    "error": resp.get("error"),
    "videos_found": len(videos),
    "sample_videos": videos[:3],  # show first 3
}, indent=2))

{
  "error": {
    "code": "ok",
    "message": "",
    "log_id": "20251210173732941E345193BDB8116FF7"
  },
  "videos_found": 3,
  "sample_videos": [
    {
      "like_count": 2,
      "share_count": 0,
      "video_description": "#navidadtiktok #parati #paratiiiiiiiiiiiiiiii #navidad ",
      "comment_count": 1,
      "id": 7581270062129007894,
      "username": "manuel.l091",
      "view_count": 269,
      "create_time": 1765151998,
      "region_code": "ES"
    },
    {
      "create_time": 1765151964,
      "like_count": 115,
      "username": "agueda715",
      "video_description": "\ud83c\udf1f\ud83c\udf84Disney\ud83c\udf84\u2708\ufe0f\ud83c\udf1f2025 Regalo de navidad  para mis hijos bellos. #navidad #merrychrismas #vida #viaje #papa ",
      "view_count": 2800,
      "comment_count": 12,
      "id": 7581269908458163478,
      "region_code": "ES",
      "share_count": 5
    },
    {
      "share_count": 0,
      "video_description": "buen\u00edsimos , permanente, te deja, efecto

**OC24: Does the platform’s API allow for filtering data based on content language?**

*This item verifies whether the API allows for applying language-based filters to data extraction. The assessment should test the endpoint for the main content type to confirm that public post data can be filtered by content language.*


In [ ]:
# This API does not support filtering data bsed on content language.


**OC25: Does the platform’s API allow for filtering data by specific time periods?**

*This item verifies whether the API allows applying temporal filters to data extraction. The assessment should test the endpoint for the main content type to confirm that public post data can be filtered by custom time ranges.*


In [196]:
from datetime import datetime
import json

if not access_token:
    raise RuntimeError("Run the auth cell first so access_token is defined.")

endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "username",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "view_count",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

tests = [
    ("20241201", "20241225"),  # Dec 1–25, 2024
    ("20251101", "20251121"),  # Nov 1–21, 2025
]

for start_date, end_date in tests:
    payload = {
        "query": {"and": [
            {"operation": "EQ", "field_name": "keyword", "field_values": ["music"]},  # change keyword as needed
        ]},
        "start_date": start_date,
        "end_date": end_date,
        "max_count": 10,
    }
    resp = request_to_api(access_token, full_url, payload)
    videos = resp.get("data", {}).get("videos", []) or []
    print(f"\nRange {start_date}–{end_date}")
    print("error:", resp.get("error"))
    print("videos_found:", len(videos))
    for v in videos[:3]:
        print("  id:", v.get("id"), "| create_time:", v.get("create_time"), "| desc:", (v.get("video_description") or "")[:80])



Range 20241201–20241225
error: {'code': 'ok', 'message': '', 'log_id': '20251210175815CFDB97C2AC1BFB1211F5'}
videos_found: 6
  id: 7452503527563283755 | create_time: 1735171196 | desc: My beautiful cuz enjoy the music 🤣🤣🤣
  id: 7452503521468910853 | create_time: 1735171197 | desc: Tu solucion soy yo  #manuelturizo #viral#music#humor
  id: 7452503517643803926 | create_time: 1735171199 | desc: #onurcanozcanaşk #askainanmayanlarasenianlattım #songslyrics #fypツ #kesfetteyiz 

Range 20251101–20251121
error: {'code': 'ok', 'message': '', 'log_id': '20251210175818A164DF502D392D11B694'}
videos_found: 10
  id: 7575332720419654930 | create_time: 1763769599 | desc: ⭐ LALI EMOCIONA CON SU MENSAJE A EROS RAMAZZOTTI #ChismesIluminados #Lali #ErosR
  id: 7575332715902274834 | create_time: 1763769599 | desc: Maneskin- gossip. 💊💊💊 . . #music #maneskin #gossip #spotyfy 
  id: 7575332713993899286 | create_time: 1763769599 | desc: Lio - Amoureuse solo #Amoureusesolo #lio_la_vraie #Lio #bananasplit #amour

### TIMELINESS

**OC26: Can data from newly published content be extracted from the platform’s API in near real time?**

*This item verifies whether the API allows the collection of data from specific content within one hour of its publication. The assessment should test the endpoint for the main content type to confirm that it allows the ready extraction of recent public posts data.*


In [ ]:
# Use this cell to develop an example where a request for posts is made.
# Describe the endpoint and the search parameters used.
# Please leave the result as the output of this cell.
# Print the most recent publication date find in the response.
endpoint_suffix = None # SET correct value (e.g "video/query")
fields = None # SET correct value (e.g "id,like_count")
full_url = f"{base_url}/{endpoint_suffix}/?fields={fields}"
payload = {} # check paylod example
response = request_to_api(access_token, full_url, payload)
collection_date = datetime.now().isoformat()
most_recent_post_date = None # SET THE CORRECT VALUE 
print("Collection date:", collection_date)
print("Most recent post:", most_recent_post_date)

In [203]:
endpoint_suffix = "video/query"
fields = ",".join([
    "id",
    "video_description",
    "username",
    "create_time",
    "region_code",
    "like_count",
    "comment_count",
    "share_count",
    "view_count",
])
full_url = f"{base_url}{endpoint_suffix}?fields={fields}"

end_date = datetime.utcnow()
start_date = end_date - timedelta(days=1)  # last 24h window (date-level granularity)

payload = {
    "query": {"and": [
        {"operation": "EQ", "field_name": "keyword", "field_values": ["climate"]},  # adjust/remove keyword as needed
    ]},
    "start_date": start_date.strftime("%Y%m%d"),
    "end_date": end_date.strftime("%Y%m%d"),
    "max_count": 20,
}

resp = request_to_api(access_token, full_url, payload)
videos = resp.get("data", {}).get("videos", []) or []
collection_date = datetime.now(timezone.utc)

def to_iso(ts):
    if isinstance(ts, (int, float)):
        return datetime.fromtimestamp(ts, tz=timezone.utc)
    try:
        return datetime.fromisoformat(str(ts).replace("Z", "+00:00"))
    except Exception:
        return None

# Find most recent create_time
recent_dt = None
if videos:
    times = [to_iso(v.get("create_time")) for v in videos]
    times = [t for t in times if t]
    if times:
        recent_dt = max(times)

print("API error:", resp.get("error"))
print("videos_found:", len(videos))
print("Collection date (UTC):", collection_date.isoformat())

if recent_dt:
    diff = collection_date - recent_dt
    print("Most recent post (UTC):", recent_dt.isoformat())
    print("Time difference (collection - most recent):", diff)
else:
    print("Most recent post: None found (no videos or missing timestamps)")


API error: {'code': 'ok', 'message': '', 'log_id': '20251210181321A782BDADE2C68012CF11'}
videos_found: 0
Collection date (UTC): 2025-12-10T18:13:22.094625+00:00
Most recent post: None found (no videos or missing timestamps)
